# 08 · Contract-demand and EV-charger rules (post-hoc additions)
Notebook 06 found two held-out label types with no rule. This notebook adds a rule for each **after the detector freeze**, in a separate module (`src/demand_ev_rules.py`);
`src/detector.py` and the frozen alerts are untouched. Both rules are defined without labels: the demand thresholds are the engineering guide's (920 kVA EV-throttle level, 1,000 kVA
contract demand), and the EV rule is "any draw above 1 kW between 01:00 and 05:00", where the pre-April data show exactly 0 kW on every night (there is no EV label before April).
**Because the gaps were seen before the rules were written, the results below are post-hoc, not clean held-out.** The demand alert is reactive (it fires once 15-min demand has crossed
920 kVA); the predictive side is the day-ahead warning in notebook 07.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import demand_ev_rules as R, evaluation as E
pd.set_option("display.width", 220, "display.max_colwidth", 150)

In [2]:
A = R.all_additions()
display(A[["rule", "start", "end", "alert_time", "flagged_h", "cost_inr", "evidence"]])
A.to_parquet(ROOT / "data/clean/alerts_demand_ev.parquet", index=False)

,rule,start,end,alert_time,flagged_h,cost_inr,evidence
0,DEMAND_NEAR_CONTRACT,2026-05-12 14:00:00,2026-05-12 17:00:00,2026-05-12 14:15:00,3.00,49880.0,"Site demand 1,086 kVA at 14:45 (BREACH); EV-throttle level 920, contract demand 1,000 kVA. Penalty if this stands as the month's MD: Rs 49,880 (ex..."
1,EV_CHARGER_NIGHT_ON,2026-05-20 01:00:00,2026-05-27 06:00:00,2026-05-20 02:00:00,40.00,NaN,EV chargers draw 6.0 kW between 01:00 and 05:00 on 8 night(s); normal for those hours is 0 kW. Likely a stuck charger; check at the charger panel....
2,EV_CHARGER_NIGHT_ON,2026-06-04 01:00:00,2026-06-07 06:00:00,2026-06-04 02:00:00,20.00,NaN,EV chargers draw 6.0 kW between 01:00 and 05:00 on 4 night(s); normal for those hours is 0 kW. Likely a stuck charger; check at the charger panel....
3,DEMAND_NEAR_CONTRACT,2026-06-18 14:00:00,2026-06-18 17:00:00,2026-06-18 14:15:00,3.00,38338.0,"Site demand 1,066 kVA at 15:30 (BREACH); EV-throttle level 920, contract demand 1,000 kVA. Penalty if this stands as the month's MD: Rs 38,338 (ex..."
4,DEMAND_NEAR_CONTRACT,2026-06-25 14:00:00,2026-06-25 17:00:00,2026-06-25 14:15:00,2.25,18386.0,"Site demand 1,032 kVA at 16:00 (BREACH); EV-throttle level 920, contract demand 1,000 kVA. Penalty if this stands as the month's MD: Rs 18,386 (ex..."


## Check against the answer key (diagnostic, post-hoc)
Labels read through `E.heldout_labels_for_diagnostics()`; nothing here changes a rule.

In [3]:
L = E.heldout_labels_for_diagnostics()
L = L[L.anomaly_type.isin(["CONTRACT_DEMAND_BREACH", "EV_CHARGER_FAULT"])]
rows = [dict(label=r.anomaly_type, start=r.start, alerts=int(((A.site == r.site) & E._overlap(A.start, A.end, r.start, r.end)).sum())) for r in L.itertuples()]
display(pd.DataFrame(rows))
fp = sum(not E._overlap(a.start, a.end, L.start, L.end).any() for a in A.itertuples())
print(f"{len(A)} alerts over the whole year; {fp} do not overlap a label of these two types. Labels covered: {sum(r['alerts'] > 0 for r in rows)} of {len(rows)}.")

,label,start,alerts
0,CONTRACT_DEMAND_BREACH,2026-05-12 14:00:00,1
1,EV_CHARGER_FAULT,2026-05-20 00:00:00,1
2,EV_CHARGER_FAULT,2026-06-04 00:00:00,1
3,CONTRACT_DEMAND_BREACH,2026-06-18 14:00:00,1
4,CONTRACT_DEMAND_BREACH,2026-06-25 14:00:00,1


5 alerts over the whole year; 0 do not overlap a label of these two types. Labels covered: 5 of 5.


**Limits.** Three demand events and two EV episodes are too few to call this validated: it shows the rules are consistent with the key, with no false alerts in the year.
The demand alert comes ~15 min after the threshold is crossed, so it informs but cannot prevent that interval's peak; prevention is the forecast's job. The ₹ shown is the tariff's
excess-demand charge if that peak stands as the month's MD (see OPEN_QUESTIONS item 2 on whether the penalty is charged on top).